In [11]:
from pyuvdata import UVData

In [6]:
uvd.get_antpairpols?

Signature: uvd.get_antpairpols()
Docstring:
Get the unique antpair + pol tuples that have data associated with them.

Returns
-------
list of tuples of int
    list of unique antpair + pol tuples (ant1, ant2, pol) with data
    associated with them.
File:      /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyuvdata/uvdata/uvdata.py
Type:      method

In [12]:
uvd=UVData()
uvd.read('../test_data/vis-eor-fgs.uvh5')
antpairpols = uvd.get_antpairpols()

Telescope Hex37-14.6m is not in known_telescopes.


In [13]:
import numpy as np
# Get data for single baseline
print("Baseline:", antpairpols[0])
d = uvd.get_data(antpairpols[0])
w = ~uvd.get_flags(antpairpols[0])

# Get no. of frequencies/times
Ntimes, Nfreqs = d.shape
print("Nfreqs:", np.unique(uvd.freq_array).size, Nfreqs)
print("Ntimes:", np.unique(uvd.lst_array).size, Ntimes)


Baseline: (0, 1, 'xx')
Nfreqs: 120 120
Ntimes: 203 203


In [21]:
freqs=uvd.freq_array[0]
lsts=uvd.lst_array

In [26]:
from astropy.units import Quantity
if not uvd.use_future_array_shapes:
    freqs = freqs[0]
freqs = Quantity(freqs, unit="Hz")
freq_str = (
    f"{freqs.min().to('MHz').value:.3f}-"
    + f"{freqs.max().to('MHz').value:.3f}MHz"
)

In [30]:
all_data_weights = []
for i_bl, antpair in enumerate(uvd.get_antpairs()):
    bl_str = f"{antpair[0]}-{antpair[1]}"
d = uvd.get_data(antpair + ("xx",), force_copy=True)

In [34]:
d.shape, freqs.shape,lsts.shape

((203, 120), (120,), (203,))

In [46]:
nm_list=(np.array([0,0]),np.array([1,1]))
np.shape(nm_list)
b_sys_past=d[nm_list]
b_sys_past=np.array([[b] for b in b_sys_past])
B_cov=b_sys_past**2*np.eye(len(b_sys_past))

In [41]:
import sys_solver as sys

h_j = sys.h_j_op(freqs=freqs, lsts=lsts, nm_list=nm_list)

In [52]:
d_f=d.flatten()

In [53]:
B_i=sys.inv_mat(B_cov)
noise_rms=0.0000001
noise=noise_rms*np.eye(np.shape(d_f)[0])
Ninv=sys.inv_mat(noise)
B_i+ np.real(h_j).T @ Ninv @ np.real(h_j) + np.imag(h_j).T @ Ninv @ np.imag(h_j)

Casting complex values to real discards the imaginary part


array([[1.00000000e+07, 2.27373675e-13],
       [1.44950718e-11, 1.00000000e+07]])

In [61]:
noise_cov=np.load('../noise-cov.npy')

In [65]:
type(noise_cov)

numpy.ndarray

5.660889025932196e-05

In [9]:
import os
os.chdir('hydra_pspec/')

In [10]:
import sys_solver as sys